In [0]:
%sql
USE CATALOG ecommerce_logistics;
USE SCHEMA sales

In [0]:
silver_items=spark.table("bronze_items")

In [0]:
%sql
DESCRIBE TABLE bronze_items

In [0]:
from pyspark.sql.functions import col,when

In [0]:
# silver_items.filter(col("price")<0).show()
silver_items.filter(col("freight_value")<0).show()

In [0]:
silver_items.withColumn("order_item_id",col("order_item_id").cast("string")).write.mode("overwrite").saveAsTable("silver_items")

In [0]:
silver_orders=spark.table("bronze_orders")

In [0]:
silver_orders.show(100,truncate=False)

In [0]:
%sql
DESCRIBE TABLE bronze_orders

In [0]:
# silver_orders.groupBy("order_status").count().show()

In [0]:
# silver_orders.filter(col("order_purchase_timestamp").isNull()).show()

In [0]:
silver_orders.filter(col("order_delivered_customer_date").isNull()).show()

In [0]:
silver_orders=silver_orders.withColumn("is_delivered",when(col("order_delivered_customer_date").isNotNull(),1).otherwise(0))

In [0]:
silver_orders=silver_orders.withColumn("delivery_time_days",when(col("is_delivered")==1,datediff(col("order_delivered_customer_date"),col("order_purchase_timestamp"))).otherwise(None))

In [0]:
silver_orders.write.mode("overwrite").saveAsTable("silver_orders")

In [0]:
silver_payments=spark.table("bronze_payments")

In [0]:
%sql
DESCRIBE TABLE bronze_payments

In [0]:
silver_payments.filter(col("payment_value").isNull()).show()

In [0]:
silver_payments.write.format("delta").mode("overwrite").saveAsTable("silver_payments")